# Choosing LLMs: answer generator and supervisor agent
Two separate tests, because the two jobs need different strengths:

- **Part A, answer generator (RAG):** does the model answer *only* from the retrieved context, and say "I don't know" when the answer isn't there?
- **Part B, supervisor agent:** does the model pick the right tools, in the right order, with the right arguments?

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

It runs for free out of the box with two small local models (Qwen2.5 1.5B and 3B) and a local 7B judge. To test real candidates, add API models in the settings cell and put their keys in Colab's **Secrets** panel (key icon on the left sidebar).

In [ ]:
!pip install -q -U openai transformers accelerate bitsandbytes pandas

## Settings: models to compare and the judge

In [ ]:
# ---- Models to compare -------------------------------------------------
# backend "hf"  = runs on the Colab GPU (free, no key). Small models only on a T4.
# backend "api" = any OpenAI-compatible API (OpenAI, OpenRouter, Groq, Together,
#                 Gemini's OpenAI endpoint, a vLLM/Ollama server...). Put the API key in
#                 Colab's Secrets panel (key icon on the left) under the name in key_name.
# price_in / price_out = $ per 1M tokens (optional, for the cost column).
MODELS = {
    "Qwen2.5-1.5B (local)": {"backend": "hf", "model": "Qwen/Qwen2.5-1.5B-Instruct"},
    "Qwen2.5-3B (local)":   {"backend": "hf", "model": "Qwen/Qwen2.5-3B-Instruct"},
    # "My API model": {"backend": "api", "base_url": "https://openrouter.ai/api/v1",
    #                  "model": "provider/model-id", "key_name": "OPENROUTER_API_KEY",
    #                  "price_in": 0.0, "price_out": 0.0},
}

# ---- Judge (grades the generator answers) ---------------------------------
# Use the strongest model you can: an API model is best. The local fallback is a
# 7B model loaded in 4-bit so it fits on a T4; it is slower and less reliable.
JUDGE = {"backend": "hf", "model": "Qwen/Qwen2.5-7B-Instruct", "quant": "4bit"}
# JUDGE = {"backend": "api", "base_url": "https://openrouter.ai/api/v1",
#          "model": "provider/strong-model-id", "key_name": "OPENROUTER_API_KEY"}

## Model client (local GPU or any OpenAI-compatible API)

In [ ]:
import os, re, json, time, gc
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_hf = {"id": None, "tok": None, "model": None}

def hf_unload():
    _hf.update(id=None, tok=None, model=None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def hf_load(cfg):
    key = (cfg["model"], cfg.get("quant"))
    if _hf["id"] == key:
        return
    hf_unload()
    kw = {"device_map": "auto"}
    if cfg.get("quant") == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    print(f"loading {cfg['model']} ...")
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    _hf.update(id=key, tok=tok, model=AutoModelForCausalLM.from_pretrained(cfg["model"], **kw))

def chat_hf(cfg, messages, max_tokens):
    hf_load(cfg)
    tok, mdl = _hf["tok"], _hf["model"]
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(mdl.device)
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(**enc, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    new = out[0, enc["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True), time.time() - t0, enc["input_ids"].shape[1], len(new)

_clients = {}

def chat_api(cfg, messages, max_tokens):
    from openai import OpenAI
    if cfg["base_url"] not in _clients:
        _clients[cfg["base_url"]] = OpenAI(base_url=cfg["base_url"], api_key=get_secret(cfg["key_name"]))
    client = _clients[cfg["base_url"]]
    kwargs = {"model": cfg["model"], "messages": messages, "max_tokens": max_tokens, "temperature": 0}
    for attempt in range(4):
        try:
            t0 = time.time()
            r = client.chat.completions.create(**kwargs)
            u = r.usage
            return (r.choices[0].message.content or ""), time.time() - t0, getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0
        except Exception as e:
            msg = str(e)
            if "max_tokens" in msg and "max_tokens" in kwargs:          # newer models want max_completion_tokens
                kwargs["max_completion_tokens"] = kwargs.pop("max_tokens")
            elif "temperature" in msg and "temperature" in kwargs:      # some reasoning models reject temperature
                kwargs.pop("temperature")
            elif attempt < 3:
                time.sleep(2 ** attempt * 3)                            # rate limit / network: back off
            else:
                raise

def chat(cfg, messages, max_tokens=300):
    """Returns (text, seconds, input_tokens, output_tokens)."""
    fn = chat_hf if cfg["backend"] == "hf" else chat_api
    text, secs, tin, tout = fn(cfg, messages, max_tokens)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()   # drop reasoning traces
    return text, secs, tin, tout

def cost_usd(cfg, tin, tout):
    return (tin * cfg.get("price_in", 0) + tout * cfg.get("price_out", 0)) / 1e6

def parse_json(text, want="object"):
    text = re.sub(r"```(?:json)?", "", text)
    open_c, close_c = ("{", "}") if want == "object" else ("[", "]")
    start, end = text.find(open_c), text.rfind(close_c)
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except Exception:
        return None

print("ready")

# Part A: answer generator
**Input:** `gen_eval.csv` with columns
- `question`
- `context`: the retrieved chunks, separated by blank lines (or already numbered `[1] … [2] …`)
- `expected_answer`: the correct answer, or `UNANSWERABLE` if the context doesn't contain it

Include **10–20% unanswerable questions**. They are the main test of faithfulness: a good generator refuses instead of answering from its own knowledge.

Use either your own file (next cell) or the demo (the cell after).

In [ ]:
from google.colab import files
uploaded = files.upload()   # gen_eval.csv with columns: question, context, expected_answer
# context = the retrieved chunks, one per line or separated by blank lines.
# expected_answer = the correct answer, or UNANSWERABLE if the context does not contain it.

### (Optional) Demo data
40 questions built from the ML handbook passages: 30 answerable (3 in Hindi, Spanish or Portuguese) and 10 where the right passage is deliberately missing.

In [ ]:
import random, re, pandas as pd

# 50 textbook-style ML passages and their questions (same data as the earlier notebooks)
PASSAGES = {
    'c01': 'The bias-variance tradeoff describes two sources of prediction error. A very simple model has high bias: it underfits because it cannot capture the true pattern. A very flexible model has high variance: its predictions change a lot with small changes in the training set. Good models balance the two so that error on unseen data is lowest.',
    'c02': 'Overfitting happens when a model memorizes noise in the training data instead of learning general patterns, so training accuracy is high but test accuracy is poor. Common remedies include collecting more data, simplifying the model, and adding regularization, which penalizes large or complex parameter values.',
    'c03': 'L1 regularization, used in Lasso regression, adds the sum of the absolute values of the weights to the loss. Because of the shape of this penalty, many weights are pushed exactly to zero, so Lasso also performs automatic feature selection.',
    'c04': 'L2 regularization, used in Ridge regression and called weight decay in neural networks, adds the sum of squared weights to the loss. It shrinks all weights smoothly toward zero but rarely makes them exactly zero, which helps when many features are correlated.',
    'c05': 'In k-fold cross-validation the dataset is split into k equal parts. The model is trained k times, each time holding out a different part for evaluation, and the k scores are averaged. This gives a more reliable estimate of performance than a single train/test split, especially on small datasets.',
    'c06': 'Data should be split into training, validation and test sets before any preprocessing is fitted. If statistics such as the mean used for scaling are computed on the full dataset, information from the test set leaks into training and the reported score becomes too optimistic. This problem is called data leakage.',
    'c07': 'Gradient descent minimizes a loss function by repeatedly moving the parameters a small step in the direction opposite to the gradient. The step size is the learning rate: if it is too small, training is very slow; if it is too large, the loss can bounce around or diverge.',
    'c08': 'Batch gradient descent uses the whole dataset for every update, which is slow for large data. Stochastic gradient descent updates after each single example, and mini-batch gradient descent uses small groups such as 32 or 256 examples. Mini-batches are the standard choice because they balance noisy updates with efficient use of GPUs.',
    'c09': 'Adam is an adaptive optimizer that keeps a running average of past gradients (momentum) and of their squared values. It uses these to give each parameter its own effective learning rate, which usually makes training converge quickly with little tuning. It is a common default optimizer for deep learning.',
    'c10': 'Linear regression predicts a numeric target as a weighted sum of input features plus an intercept. The weights are usually found by ordinary least squares, which chooses the line or plane that minimizes the sum of squared differences between predicted and actual values.',
    'c11': 'Logistic regression is used for binary classification. It passes a linear combination of the features through the sigmoid function, which squashes any number into a probability between 0 and 1. It is trained by minimizing log loss, also called binary cross-entropy.',
    'c12': 'Precision is the fraction of predicted positives that are actually positive, while recall is the fraction of actual positives that the model found. In spam filtering you may favor precision so real emails are not lost; in disease screening you favor recall so few sick patients are missed. The F1 score is the harmonic mean of the two.',
    'c13': 'The ROC curve plots the true positive rate against the false positive rate as the classification threshold changes. The area under this curve, AUC, summarizes performance across all thresholds: 0.5 means random guessing and 1.0 means perfect separation of the classes.',
    'c14': "Accuracy can be misleading when classes are imbalanced. If only 1% of transactions are fraud, a model that always predicts 'not fraud' scores 99% accuracy while being useless. A confusion matrix, which counts true positives, false positives, true negatives and false negatives, shows what is really happening.",
    'c15': 'To handle imbalanced classes you can oversample the minority class, undersample the majority class, or create synthetic minority examples with SMOTE. Another option is to keep the data as is and give a higher class weight to the rare class in the loss function.',
    'c16': 'A decision tree splits the data with a sequence of yes/no questions on the features. At each node it chooses the split that makes the child groups as pure as possible, measured by Gini impurity or entropy. Trees are easy to interpret but grow deep and overfit unless their depth is limited or they are pruned.',
    'c17': 'A random forest trains many decision trees on bootstrap samples of the data, a technique called bagging, and also considers only a random subset of features at each split. Averaging the votes of these diverse trees reduces variance and makes the forest far more robust than a single tree.',
    'c18': 'Gradient boosting builds trees sequentially: each new small tree is trained to correct the errors, or residuals, of the ensemble built so far. Libraries such as XGBoost, LightGBM and CatBoost implement this efficiently and are often the strongest choice for tabular data competitions.',
    'c19': 'A support vector machine finds the boundary that separates two classes with the widest possible margin. Only the points closest to the boundary, the support vectors, define it. With the kernel trick, an SVM can learn curved boundaries by implicitly mapping data into a higher-dimensional space.',
    'c20': 'The k-nearest neighbors algorithm has no real training step. To classify a new point it finds the k most similar points in the training data and takes a majority vote of their labels. It is simple but becomes slow on large datasets and is sensitive to the scale of the features.',
    'c21': "Naive Bayes classifiers apply Bayes' theorem while assuming that features are independent of each other given the class. The assumption is rarely true, yet the method works surprisingly well for text tasks such as spam detection and is extremely fast to train.",
    'c22': 'K-means clustering groups unlabeled data into k clusters. It starts with k centroids, assigns each point to the nearest centroid, moves each centroid to the mean of its assigned points, and repeats until assignments stop changing. The elbow method helps choose a reasonable k.',
    'c23': 'Principal component analysis reduces the number of features by projecting the data onto new axes, the principal components, that capture the most variance. The first component points in the direction where the data varies most. PCA is used to compress data, remove noise and speed up other algorithms.',
    'c24': 't-SNE and UMAP are non-linear techniques for visualizing high-dimensional data in two or three dimensions. They try to keep points that are close in the original space close in the plot. They are great for exploring clusters, but distances between far-apart groups in the plot should not be over-interpreted.',
    'c25': 'Many algorithms, including gradient descent, k-nearest neighbors and SVMs, work poorly when features are on very different scales. Standardization rescales a feature to mean 0 and standard deviation 1, while min-max normalization squeezes it into the range 0 to 1. Tree-based models generally do not need scaling.',
    'c26': 'Most models need numbers, so categorical features such as city or color must be encoded. One-hot encoding creates a separate 0/1 column for every category. When a feature has thousands of categories, alternatives such as target encoding or learned embeddings are more practical.',
    'c27': 'Real datasets often have missing values. Simple strategies are dropping rows or filling gaps with the mean, median or most frequent value. More advanced imputation predicts the missing value from other features, and adding a flag column that marks where data was missing can itself be informative.',
    'c28': 'An artificial neuron computes a weighted sum of its inputs and passes it through a non-linear activation function. Without non-linearity, stacking layers would still produce only a linear model. ReLU, which outputs zero for negative inputs and the input itself otherwise, is the most common activation in hidden layers.',
    'c29': 'Backpropagation computes the gradient of the loss with respect to every weight in a neural network. It runs a forward pass to get predictions, then applies the chain rule backwards from the output layer to the input layer. These gradients are then used by an optimizer to update the weights.',
    'c30': 'In very deep networks, gradients can shrink as they are multiplied through many layers, so early layers barely learn; this is the vanishing gradient problem. Sigmoid activations make it worse. ReLU activations, careful weight initialization, residual connections and normalization layers all help.',
    'c31': 'Dropout is a regularization technique for neural networks. During training it randomly switches off a fraction of neurons, for example 20 to 50 percent, at each step. This prevents neurons from relying too much on each other. At inference time all neurons are active.',
    'c32': 'Batch normalization normalizes the activations of a layer using the mean and variance of the current mini-batch, then applies a learned scale and shift. It stabilizes and speeds up training and allows higher learning rates. Layer normalization, used in transformers, normalizes across the features of a single example instead.',
    'c33': 'Convolutional neural networks are designed for images. Convolution layers slide small learnable filters over the image to detect local patterns such as edges and textures, and pooling layers shrink the feature maps. Deeper layers combine simple patterns into complex shapes like eyes or wheels.',
    'c34': 'Recurrent neural networks process sequences one step at a time while carrying a hidden state that summarizes what they have seen. Plain RNNs struggle with long sequences, so LSTM and GRU cells add gates that decide what information to keep or forget over many time steps.',
    'c35': 'The transformer architecture replaces recurrence with self-attention: every token computes how much to attend to every other token, so relationships across a whole sentence are captured in parallel. Positional encodings add word order. Transformers are the foundation of modern language models such as BERT and GPT.',
    'c36': 'Word embeddings represent words as dense vectors so that words used in similar contexts get similar vectors. Word2vec learns them by predicting a word from its neighbors or the neighbors from the word. The famous example is that king minus man plus woman lands close to queen.',
    'c37': 'Transfer learning reuses a model pretrained on a large dataset for a new task. You typically freeze the early layers, which capture general features, and fine-tune the later layers on your smaller dataset. This gives strong results with far less data and compute than training from scratch.',
    'c38': 'Early stopping monitors the validation loss during training and stops when it has not improved for a set number of epochs, called the patience. The weights from the best epoch are restored. It is a simple and effective guard against overfitting in iterative training.',
    'c39': 'Hyperparameters, such as learning rate or tree depth, are set before training rather than learned. Grid search tries every combination on a fixed grid, random search samples combinations and is often more efficient, and Bayesian optimization tools like Optuna use past results to choose promising settings next.',
    'c40': 'As the number of features grows, the volume of the space grows so fast that data points become sparse and far apart. Distance-based methods lose meaning because all points look almost equally distant. This curse of dimensionality is why feature selection and dimensionality reduction matter.',
    'c41': 'In reinforcement learning an agent interacts with an environment, takes actions, and receives rewards. Its goal is to learn a policy that maximizes total future reward. Q-learning estimates the value of each action in each state, and the agent must balance exploring new actions with exploiting known good ones.',
    'c42': 'A generative adversarial network trains two networks against each other. The generator creates fake samples from random noise, and the discriminator tries to tell fake from real. As both improve, the generator learns to produce realistic images. Training can be unstable, and mode collapse is a common failure.',
    'c43': 'An autoencoder compresses its input into a small latent representation with an encoder and reconstructs it with a decoder. It is used for dimensionality reduction, denoising and anomaly detection, since unusual inputs reconstruct poorly. A variational autoencoder makes the latent space smooth so new samples can be generated.',
    'c44': 'Regression models are evaluated with error metrics. Mean absolute error is the average size of mistakes, while root mean squared error penalizes large mistakes more heavily. R-squared shows the share of variance in the target explained by the model, where 1 is a perfect fit.',
    'c45': 'For multi-class classification, the final layer produces one score per class, and the softmax function turns these scores into probabilities that sum to 1. The model is trained with categorical cross-entropy loss, which heavily penalizes confident wrong predictions.',
    'c46': 'Data augmentation creates new training examples by applying label-preserving changes to existing ones. For images this includes flips, rotations, crops and color changes; for text it includes synonym replacement or back-translation. It effectively enlarges the dataset and improves generalization.',
    'c47': "After deployment, a model's performance can decay because the incoming data changes over time, which is known as data drift or concept drift. Production systems should monitor input distributions and prediction quality, and retrain the model when significant drift is detected.",
    'c48': "Maximum likelihood estimation picks the parameters that make the observed data most probable. Maximum a posteriori estimation also multiplies in a prior belief about the parameters using Bayes' theorem. With a Gaussian prior, MAP estimation turns out to be equivalent to L2 regularization.",
    'c49': 'Learning curves plot training and validation error as the training set grows. If both errors are high and close together, the model underfits and needs more capacity. If training error is low but validation error stays much higher, the model overfits and more data or regularization will help.',
    'c50': 'Retrieval-augmented generation combines a language model with a search step. Documents are split into chunks, converted to embeddings and stored in a vector database. At question time the most similar chunks are retrieved and given to the model as context, which reduces hallucination and lets it answer from private data.',
}

QUESTIONS = {
    'c01': ['why do a model that is too simple and one that is too complex both perform badly?', 'what is the tradeoff between underfitting and overfitting called'],
    'c02': ["my model scores 99% on training data but badly on new data, what's going on", 'जब मॉडल ट्रेनिंग डेटा का शोर याद कर लेता है तो उसे क्या कहते हैं?'],
    'c03': ['which penalty makes some coefficients become exactly zero', 'regularization method that also picks features for you'],
    'c04': ['what is weight decay in neural nets', 'which regularization shrinks coefficients but keeps all features, good for correlated inputs'],
    'c05': ['how to get a more reliable accuracy estimate when I only have a small dataset', '¿cómo funciona la validación cruzada con k particiones?'],
    'c06': ['why is my test score unrealistically high, I scaled the data before splitting', 'should I fit the scaler on all the data or only on the training set'],
    'c07': ['my loss keeps exploding during training, what setting should I lower', 'how does the basic optimization algorithm for training models work'],
    'c08': ['why do people train with batches of 32 instead of the full dataset', 'difference between batch, stochastic and mini-batch updates'],
    'c09': ['which optimizer should I use by default for a neural network', 'optimizer that gives every weight its own learning rate using momentum'],
    'c10': ['simplest model to predict house prices from size and number of rooms', 'what does ordinary least squares minimize'],
    'c11': ['which classic model outputs a probability for yes/no problems', 'what does the sigmoid function do'],
    'c12': ['for detecting cancer, what matters more: catching every case or avoiding false alarms', 'what is F1 score'],
    'c13': ['what does an AUC of 0.5 mean', 'como avaliar um classificador em todos os limiares de decisão?'],
    'c14': ['my fraud model has 99 percent accuracy but catches nothing, why', 'table showing true and false positives and negatives'],
    'c15': ['how to fix a dataset where one class is very rare', 'what is SMOTE'],
    'c16': ['model that makes predictions using a flowchart of if-else questions', 'what is Gini impurity used for'],
    'c17': ['how does bagging reduce overfitting of trees', 'why is a forest of trees better than one tree'],
    'c18': ['best algorithm for Kaggle tabular data', 'ensemble where each new tree fixes the mistakes of the previous ones'],
    'c19': ['classifier that maximizes the gap between classes', "how can a linear classifier separate data that isn't linearly separable"],
    'c20': ['lazy learning algorithm that looks at the closest examples to decide', 'which algorithm just votes among similar samples'],
    'c21': ['fast simple classifier for spam emails based on word probabilities', 'why is this Bayes classifier called naive'],
    'c22': ['how to group customers into segments without labels', 'how do I choose the number of clusters'],
    'c23': ['how to reduce 100 features to 10 while keeping most of the information', 'técnica para reduzir a dimensionalidade dos dados mantendo a variância'],
    'c24': ['how to make a 2D picture of my embeddings to see clusters', 'is the distance between clusters in a t-SNE plot meaningful'],
    'c25': ['do I need to normalize features for a random forest', 'one feature is in thousands and another in decimals, is that a problem'],
    'c26': ['how to feed a column like country name into a model', 'what to do with a categorical feature that has thousands of values'],
    'c27': ['some cells in my dataset are empty, how should I handle them', '¿con qué valor relleno los datos faltantes?'],
    'c28': ['why do neural networks need activation functions', 'what does ReLU do'],
    'c29': ['how does a neural network know how to adjust each weight', 'chain rule in training deep nets'],
    'c30': ['why do the first layers of my deep network stop learning', 'what problem do residual connections solve'],
    'c31': ['randomly turning off neurons during training, what is this called', 'is dropout used at prediction time'],
    'c32': ['difference between batch norm and layer norm', 'technique that normalizes layer outputs to speed up training'],
    'c33': ['what type of network is used for image classification', 'what do the filters learn in the early layers'],
    'c34': ['neural network with memory for time series', 'why were LSTMs invented'],
    'c35': ['what architecture is behind GPT', '¿cómo funciona la autoatención en los transformers?'],
    'c36': ['king minus man plus woman equals queen', 'how are words turned into vectors'],
    'c37': ['I only have 500 labeled images, can I still train a good model', 'fine-tuning a pretrained network'],
    'c38': ['when should I stop training, how many epochs', 'validation loss started going up while training loss keeps going down'],
    'c39': ['how to find the best learning rate and tree depth automatically', 'grid search vs random search'],
    'c40': ['why does kNN get worse with hundreds of features', 'what happens to distances in very high dimensional spaces'],
    'c41': ['how do AIs learn to play games through trial and error', 'exploration vs exploitation'],
    'c42': ['two networks competing, one creates fakes and one detects them', 'what is mode collapse'],
    'c43': ['detect anomalies using reconstruction error', 'what is a VAE'],
    'c44': ['RMSE vs MAE, which one should I report', 'what does an R squared of 0.8 mean'],
    'c45': ['how does a network output probabilities for 10 classes', 'loss function for classifying digits 0-9'],
    'c46': ['how to get more training images without collecting new data', 'back-translation for text data'],
    'c47': ['my model was great at launch but got worse months later', 'मॉडल को दोबारा कब ट्रेन करना चाहिए?'],
    'c48': ['difference between MLE and MAP', 'connection between priors and regularization'],
    'c49': ['how to tell if more data will help my model', 'plot of error versus training set size'],
    'c50': ['how can a chatbot answer questions from my company documents', 'why store document chunks in a vector database'],
}

def key_sentence(passage, question):
    sents = re.split(r"(?<=[.!?])\s+", passage)
    qw = {w for w in re.findall(r"\w+", question.lower()) if len(w) > 3}
    best = max(sents, key=lambda s: len(qw & set(re.findall(r"\w+", s.lower()))))
    return best if qw & set(re.findall(r"\w+", best.lower())) else sents[0]

rng = random.Random(7)
ids = list(PASSAGES)
rows = []

def make_context(include=None, exclude=None):
    pool = [i for i in ids if i not in (include, exclude)]
    chosen = rng.sample(pool, 3 if include is None else 2)
    if include:
        chosen.insert(rng.randrange(3), include)
    return "\n\n".join(PASSAGES[i] for i in chosen)

# 30 answerable: the right passage is one of the 3 in the context (3 of them in Hindi/Spanish/Portuguese)
for cid in ids[:27]:
    q = QUESTIONS[cid][0]
    rows.append({"question": q, "context": make_context(include=cid), "expected_answer": key_sentence(PASSAGES[cid], q)})
for cid in ["c02", "c05", "c13"]:
    q = QUESTIONS[cid][1]
    rows.append({"question": q, "context": make_context(include=cid), "expected_answer": key_sentence(PASSAGES[cid], q)})

# 10 unanswerable: the right passage is left out on purpose. The model knows ML, so it will be
# tempted to answer from its own knowledge - a faithful model must say it doesn't know.
for cid in ids[30:40]:
    rows.append({"question": QUESTIONS[cid][0], "context": make_context(exclude=cid), "expected_answer": "UNANSWERABLE"})

pd.DataFrame(rows).to_csv("gen_eval.csv", index=False)
print(f"gen_eval.csv created: {len(rows)} questions (10 unanswerable)")

### A1. Generate answers with every model

In [ ]:
SYSTEM_GEN = """You answer questions using ONLY the numbered context passages.
Rules:
- Use only facts stated in the context. Do not add outside knowledge, even if you know the topic.
- Cite the passages you used, like [1] or [2][3].
- If the context does not contain the answer, reply exactly (in English): I don't know based on the provided documents.
- Otherwise answer in the same language as the question, in under 80 words."""

REFUSAL = "i don't know based on the provided documents"

def split_passages(context):
    parts = [p.strip() for p in re.split(r"\n\s*\n|\n(?=\[\d+\])", str(context)) if p.strip()]
    return [re.sub(r"^\[\d+\]\s*", "", p) for p in parts]

def gen_messages(question, context):
    ctx = "\n\n".join(f"[{i + 1}] {p}" for i, p in enumerate(split_passages(context)))
    return [{"role": "system", "content": SYSTEM_GEN},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {question}"}]

gen_df = pd.read_csv("gen_eval.csv")
gen_df["answerable"] = gen_df["expected_answer"].astype(str).str.strip().str.upper() != "UNANSWERABLE"
print(f"{len(gen_df)} questions ({(~gen_df['answerable']).sum()} unanswerable)")

gen_rows = []
for name, cfg in MODELS.items():
    print("generating with", name)
    for _, r in gen_df.iterrows():
        text, secs, tin, tout = chat(cfg, gen_messages(r["question"], r["context"]), max_tokens=200)
        gen_rows.append({"model": name, "question": r["question"], "context": r["context"],
                         "expected": r["expected_answer"], "answerable": r["answerable"], "answer": text,
                         "seconds": secs, "tokens_out": tout, "cost": cost_usd(cfg, tin, tout)})
gen_out = pd.DataFrame(gen_rows)
hf_unload()
gen_out[["model", "question", "answer"]].head(6)

### A2. Grade with the judge and compare
- **correct_%**: answerable questions answered correctly (partial = half).
- **faithful_%**: answers where every claim is supported by the context. The most important column.
- **correct_refusal_%**: unanswerable questions where the model said it doesn't know (higher is better).
- **false_refusal_%**: answerable questions it wrongly refused (lower is better).
- **cites_sources_%**: answers that include [n] citations.
- **judge_errors**: judge replies that couldn't be parsed; if high, use a stronger judge.

The judge is an LLM too, so spot-check a few of its grades in the next cell.

In [ ]:
JUDGE_PROMPT = """You are grading an answer produced by a RAG system.

CONTEXT PASSAGES:
{context}

QUESTION: {question}
EXPECTED ANSWER: {expected}
SYSTEM ANSWER: {answer}

Grade two things:
1. faithful: true if EVERY factual claim in the system answer is supported by the context passages. Claims from outside knowledge count as unsupported even if they are true. An answer that says it doesn't know is faithful.
2. correctness: "correct" if the answer conveys the expected answer's key point, "partial" if it is incomplete, "wrong" if it is incorrect or off-topic, "refused" if it says it doesn't know. If the expected answer is UNANSWERABLE, use "refused" when the system declined and "wrong" when it answered anyway.

Reply with ONLY this JSON:
{{"faithful": true, "unsupported_claims": [], "correctness": "correct"}}"""

def judge(row):
    prompt = JUDGE_PROMPT.format(context=row["context"], question=row["question"],
                                 expected=row["expected"], answer=row["answer"])
    text, *_ = chat(JUDGE, [{"role": "user", "content": prompt}], max_tokens=250)
    return parse_json(text, "object")

grades = []
for i, row in gen_out.iterrows():
    g = judge(row) or {}
    grades.append({"faithful": g.get("faithful"), "correctness": str(g.get("correctness", "judge_error")).lower(),
                   "unsupported": g.get("unsupported_claims", [])})
    if i % 20 == 0:
        print(f"judged {i + 1}/{len(gen_out)}")
gen_out = pd.concat([gen_out.reset_index(drop=True), pd.DataFrame(grades)], axis=1)
hf_unload()

def refused(row):
    return REFUSAL in row["answer"].lower() or row["correctness"] == "refused"

gen_out["refused"] = gen_out.apply(refused, axis=1)
gen_out["cited"] = gen_out["answer"].str.contains(r"\[\d+\]")

def summarize(g):
    ans, una = g[g["answerable"]], g[~g["answerable"]]
    given = g[~g["refused"]]
    score = ans["correctness"].map({"correct": 1, "partial": 0.5}).fillna(0)
    return pd.Series({
        "correct_%": round(100 * score.mean(), 1),
        "faithful_%": round(100 * given["faithful"].eq(True).mean(), 1) if len(given) else None,
        "correct_refusal_%": round(100 * una["refused"].mean(), 1) if len(una) else None,
        "false_refusal_%": round(100 * ans["refused"].mean(), 1),
        "cites_sources_%": round(100 * given["cited"].mean(), 1) if len(given) else None,
        "avg_seconds": round(g["seconds"].mean(), 2),
        "avg_tokens_out": round(g["tokens_out"].mean()),
        "cost_per_1k_questions_$": round(1000 * g["cost"].mean(), 3),
        "judge_errors": int((g["correctness"] == "judge_error").sum()),
    })

gen_report = gen_out.groupby("model").apply(summarize).sort_values("faithful_%", ascending=False)
gen_report

In [ ]:
for name, g in gen_out.groupby("model"):
    bad = g[(g["faithful"] == False) | (~g["answerable"] & ~g["refused"]) | (g["answerable"] & g["refused"])]
    print(f"\n=== {name}: {len(bad)} problem answers ===")
    for _, r in bad.head(5).iterrows():
        kind = "answered an unanswerable question" if not r["answerable"] and not r["refused"] else \
               "refused although the answer was in the context" if r["answerable"] and r["refused"] else "unfaithful"
        print(f"- [{kind}] Q: {r['question']}\n  A: {r['answer'][:220]}\n  unsupported: {r['unsupported']}")

# Part B: supervisor agent
The supervisor sees the tool list and returns a **plan as JSON**: the tools to call, in order, with arguments. Each case has the expected tool sequence and key argument values.

Replace `TOOLS` and `SUP_CASES` with your own agents and 30–50 real requests. Include cases where information is missing (the right answer is `ask_user`) and multi-step requests.

This tests the *routing and planning decision* the same way for every model. It doesn't test a provider's native function-calling API format, so once you pick a model, verify it once with your real framework (LangGraph, etc.).

In [ ]:
TOOLS = {
    "search_knowledge_base": {"description": "Search internal company documents and policies.", "args": {"query": "string"}},
    "get_sales_metrics":     {"description": "Get business numbers such as revenue, signups, churn or sales.",
                              "args": {"metric": "string", "period": "string", "region": "string (optional)"}},
    "create_task":           {"description": "Create a task for a person.", "args": {"title": "string", "assignee": "string", "due_date": "string"}},
    "send_email":            {"description": "Send an email.", "args": {"to": "email address", "subject": "string", "body": "string"}},
    "web_search":            {"description": "Search the public internet for current or external information.", "args": {"query": "string"}},
    "ask_user":              {"description": "Ask the user a clarifying question when required information is missing.", "args": {"question": "string"}},
    "answer_directly":       {"description": "Reply without tools: greetings, thanks, simple math or general questions.", "args": {"answer": "string"}},
}

# expected: the tool sequence; checks: {step_index: {arg: [accepted substrings]}}
# an arg name like "subject|body" passes if ANY of those args contains a substring.
SUP_CASES = [
    ("What's our refund policy for annual plans?", ["search_knowledge_base"], {0: {"query": ["refund"]}}),
    ("How many new signups did we get in Brazil last month?", ["get_sales_metrics"], {0: {"metric": ["signup"], "region": ["brazil"]}}),
    ("Create a task for Priya to update the pricing page by Friday", ["create_task"], {0: {"assignee": ["priya"], "title": ["pricing"], "due_date": ["fri"]}}),
    ("Email john@acme.com that the contract is ready", ["send_email"], {0: {"to": ["john@acme.com"], "subject|body": ["contract"]}}),
    ("Send an email to the client about the delay", ["ask_user"], {}),
    ("What's the latest news about the EU AI Act?", ["web_search"], {0: {"query": ["ai act"]}}),
    ("Hi! How are you?", ["answer_directly"], {}),
    ("What is 15% of 2,400?", ["answer_directly"], {0: {"answer": ["360"]}}),
    ("Find our onboarding checklist and create a task for Maria to complete it by Monday", ["search_knowledge_base", "create_task"], {0: {"query": ["onboarding"]}, 1: {"assignee": ["maria"]}}),
    ("Check last quarter's revenue in Europe and email the numbers to cfo@acme.com", ["get_sales_metrics", "send_email"], {0: {"metric": ["revenue"], "region": ["europ"]}, 1: {"to": ["cfo@acme.com"]}}),
    ("Look up what competitors charge for similar plans online", ["web_search"], {0: {"query": ["competitor", "pric"]}}),
    ("Create a task", ["ask_user"], {}),
    ("What does our leave policy say about sick days?", ["search_knowledge_base"], {0: {"query": ["sick"]}}),
    ("How did churn change in Q2?", ["get_sales_metrics"], {0: {"metric": ["churn"], "period": ["q2"]}}),
    ("Remind Alex to send the invoice by the 30th", ["create_task"], {0: {"assignee": ["alex"], "title": ["invoice"]}}),
    ("Search the web for the current USD to INR exchange rate", ["web_search"], {0: {"query": ["inr"]}}),
    ("Get this year's revenue by region, then create a task for the sales team to review it", ["get_sales_metrics", "create_task"], {0: {"metric": ["revenue"]}, 1: {"assignee": ["sales"]}}),
    ("Can you email the team?", ["ask_user"], {}),
    ("Thanks, that's all", ["answer_directly"], {}),
    ("Check if our security policy allows personal laptops and email it@acme.com to request a company laptop", ["search_knowledge_base", "send_email"], {0: {"query": ["laptop"]}, 1: {"to": ["it@acme.com"]}}),
    ("पिछले महीने भारत में कितनी बिक्री हुई?", ["get_sales_metrics"], {0: {"region": ["india", "भारत"]}}),
    ("Crie uma tarefa para o Pedro revisar o contrato até sexta", ["create_task"], {0: {"assignee": ["pedro"]}}),
    ("Who won yesterday's football final?", ["web_search"], {}),
    ("Summarize our Q3 marketing plan document", ["search_knowledge_base"], {0: {"query": ["q3", "marketing"]}}),
]
print(len(SUP_CASES), "supervisor test cases")

### B1. Run and compare
- **task_success_pct**: right tools, right order, right key arguments. The main metric.
- **right_tools_in_order_pct / right_first_tool_pct**: where it went wrong, if not.
- **valid_json_pct**: can your code parse its output at all?

In [ ]:
tool_text = "\n".join(f"- {n}: {t['description']} Arguments: {json.dumps(t['args'])}" for n, t in TOOLS.items())
SYSTEM_SUP = f"""You are a supervisor agent. Decide which tools to call, in order, to complete the user's request.

Available tools:
{tool_text}

Rules:
- Plan ALL the steps needed, in the order they must run.
- If required information is missing (for example who to email or what the task is), return only one ask_user step.
- For greetings, thanks, or simple math and general questions, return only one answer_directly step.
- Keep dates as the user wrote them (for example "Friday").
- Reply with ONLY a JSON list, no other text, like:
[{{"tool": "search_knowledge_base", "arguments": {{"query": "refund policy"}}}}]"""

def arg_ok(step, arg, accepted):
    args = step.get("arguments", {}) if isinstance(step, dict) else {}
    values = " ".join(str(args.get(a, "")) for a in arg.split("|")).lower()
    return any(s.lower() in values for s in accepted)

def score_case(plan, expected, checks):
    if isinstance(plan, dict):
        plan = plan.get("steps", [plan])
    if not isinstance(plan, list) or not all(isinstance(s, dict) for s in plan):
        return {"valid": False, "first_ok": False, "sequence_ok": False, "args_ok": False, "tools": None}
    tools = [s.get("tool") for s in plan]
    seq = tools == expected
    args = seq and all(arg_ok(plan[i], a, acc) for i, c in checks.items() for a, acc in c.items())
    return {"valid": all(t in TOOLS for t in tools), "first_ok": bool(tools) and tools[0] == expected[0],
            "sequence_ok": seq, "args_ok": args, "tools": tools}

sup_rows = []
for name, cfg in MODELS.items():
    print("supervisor test with", name)
    for request, expected, checks in SUP_CASES:
        text, secs, tin, tout = chat(cfg, [{"role": "system", "content": SYSTEM_SUP},
                                           {"role": "user", "content": request}], max_tokens=300)
        s = score_case(parse_json(text, "list"), expected, checks)
        sup_rows.append({"model": name, "request": request, "expected": expected, "raw": text,
                         "seconds": secs, "cost": cost_usd(cfg, tin, tout), **s})
sup_out = pd.DataFrame(sup_rows)
hf_unload()

sup_report = sup_out.groupby("model").agg(
    task_success_pct=("args_ok", lambda x: round(100 * x.mean(), 1)),
    right_tools_in_order_pct=("sequence_ok", lambda x: round(100 * x.mean(), 1)),
    right_first_tool_pct=("first_ok", lambda x: round(100 * x.mean(), 1)),
    valid_json_pct=("valid", lambda x: round(100 * x.mean(), 1)),
    avg_seconds=("seconds", lambda x: round(x.mean(), 2)),
    cost_per_1k_requests_usd=("cost", lambda x: round(1000 * x.mean(), 3)),
).sort_values("task_success_pct", ascending=False)
sup_report

In [ ]:
for name, g in sup_out.groupby("model"):
    bad = g[~g["args_ok"]]
    print(f"\n=== {name}: {len(bad)} failed tasks ===")
    for _, r in bad.head(8).iterrows():
        why = "invalid JSON" if r["tools"] is None else "wrong tools" if not r["sequence_ok"] else "wrong arguments"
        print(f"- [{why}] {r['request']}\n  expected {r['expected']}, got {r['tools']}\n  raw: {r['raw'][:160]!r}")

# How to decide
1. **Generator:** shortlist models with high **faithful_%** and **correct_refusal_%**, then pick the cheapest and fastest with acceptable **correct_%**. A model that answers everything but invents facts is worse than one that sometimes says "I don't know".
2. **Supervisor:** pick by **task_success_pct**. Its mistakes spread to every later step, so this is where a stronger model is usually worth the cost.
3. The same model doesn't have to do both jobs. A strong supervisor plus a cheaper generator is a common setup.
4. Always read the failure lists: they tell you if a problem is fixable with a better prompt or needs a better model.